# SupplyChainSleuth_DataIngestion



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = '68unbxgzaaxeav'
os.environ['DataZoneDomainId'] = 'dzd-574rwuctwgq31j'
os.environ['DataZoneEnvironmentId'] = 'dt8hx568pmkt2f'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "68unbxgzaaxeav",
                "DataZoneDomainId": "dzd-574rwuctwgq31j",
                "DataZoneEnvironmentId": "dt8hx568pmkt2f",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
import sys
import subprocess

def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"Successfully installed {package}")
    except Exception as e:
        print(f"Could not install {package}: {e}")

install_package('newspaper3k')

In [0]:
import sys
import subprocess

def fix_newspaper_dependencies():
    packages = ['lxml_html_clean', 'newspaper3k']
    for package in packages:
        try:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
            print(f"Successfully installed {package}")
        except Exception as e:
            print(f"Failed to install {package}: {e}")

fix_newspaper_dependencies()

In [0]:
import boto3
import json
import hashlib
import time
import requests
from newspaper import Article
from botocore.exceptions import ClientError

SECRET_NAME = "SupplyChainSleuth/NewsData"
REGION_NAME = "us-east-1" 
TABLE_NAME = "SupplyChainArticles"

session = boto3.session.Session()
secrets_client = session.client('secretsmanager', region_name=REGION_NAME)
dynamodb = session.resource('dynamodb', region_name=REGION_NAME)
table = dynamodb.Table(TABLE_NAME)

try:
    response = secrets_client.get_secret_value(SecretId=SECRET_NAME)
    secret_dict = json.loads(response['SecretString'])
    api_key = secret_dict.get('newsdata_api')
except Exception as e:
    print(f"Error: Could not find secret {SECRET_NAME} in {REGION_NAME}. {e}")
    raise e

search_categories = {
    0: [
        '"forced labor"', '"child labor"', '"human trafficking"', 
        '"modern slavery"', '"unpaid wages"', '"debt bondage"',
        '"withholding passports"', '"factory fire safety violation"',
        '"workplace fatality"', '"hazardous working conditions"',
        '"illegal mining"', '"sanctions evasion"', '"bribery scandal"'
    ],
    1: [
        '"supply chain disruption"', '"environmental fine"', 
        '"deforestation report"', '"carbon emissions violation"', 
        '"regulatory probe"', '"safety audit failure"', 
        '"labor strike"', '"union busting allegation"',
        '"product recall"', '"toxic waste leak"', 
        '"greenwashing lawsuit"', '"supplier bankruptcy"'
    ],
    2: [
        '"corporate social responsibility"', '"sustainability report"', 
        '"quarterly earnings"', '"annual general meeting"', 
        '"new warehouse opening"', '"business partnership"', 
        '"charity donation"', '"stock market update"',
        '"ceo interview"', '"product launch"', 
        '"renewable energy transition"', '"employee benefits update"'
    ]
}

def fetch_full_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        if len(article.text) > 350:
            return article.text
        return None
    except Exception:
        return None

print(f"Starting Expanded General Collection in {REGION_NAME}...")
request_count = 0
total_added = 0

for label, keywords in search_categories.items():
    for query in keywords:
        if request_count >= 195:
            print("Approaching NewsData.io limit of 200 requests.")
            break
            
        url = f"https://newsdata.io/api/1/news?apikey={api_key}&q={query}&language=en"
        
        try:
            api_response = requests.get(url)
            data = api_response.json()
            request_count += 1
            
            if data.get('status') == 'success':
                results = data.get('results', [])
                for item in results:
                    link = item.get('link')
                    if not link:
                        continue
                    
                    full_body = fetch_full_text(link)
                    
                    if not full_body:
                        continue
                        
                    article_id = hashlib.md5(full_body.encode('utf-8')).hexdigest()
                    
                    try:
                        table.put_item(
                            Item={
                                'ArticleId': article_id,
                                'Text': full_body,
                                'Label': label,
                                'Url': link,
                                'Brand': 'General_Training_Expanded',
                                'Source': 'NewsData_Expanded_Scraper'
                            },
                            ConditionExpression='attribute_not_exists(ArticleId)'
                        )
                        total_added += 1
                    except ClientError:
                        pass
                        
            elif "RateLimitExceeded" in str(data):
                print("Stop: NewsData.io rate limit reached.")
                request_count = 200
                break
                
        except Exception as e:
            print(f"Error on query {query}: {e}")

        time.sleep(2)
    
    if request_count >= 195:
        break

print(f"Finished. Total requests: {request_count} | Articles added to {REGION_NAME}: {total_added}")

In [0]:
import boto3

TABLE_NAME = "SupplyChainArticles"
REGION = "us-east-1"

def get_live_count():
    dynamodb = boto3.resource('dynamodb', region_name=REGION)
    table = dynamodb.Table(TABLE_NAME)
    
    count = 0
    response = table.scan(Select='COUNT')
    count += response.get('Count', 0)
    
    while 'LastEvaluatedKey' in response:
        response = table.scan(Select='COUNT', ExclusiveStartKey=response['LastEvaluatedKey'])
        count += response.get('Count', 0)
        
    print(f"Live Article Count: {count}")
    return count

get_live_count()

In [0]:
import boto3

# Configuration
region = "us-east-1"
table_name = "SupplyChainArticles"

dynamodb = boto3.resource("dynamodb", region_name=region)
table = dynamodb.Table(table_name)

print(f"Clearing items from {table_name}")

response = table.scan(ProjectionExpression="ArticleId")
items = response.get('Items', [])

with table.batch_writer() as batch:
    for item in items:
        batch.delete_item(Key={'ArticleId': item['ArticleId']})

print(f"Deleted {len(items)} articles")

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()